# PJM Load Forecasting — End-to-End Pipeline Walkthrough

Runs every stage of the `pjm_load_forecast` pipeline directly (the same functions
`src/dags/dag_factory.py` wires into the Airflow DAG), against the real
[`PJME_hourly.csv`](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)
dataset (~145k hourly rows, 2002–2018) staged at `data/pjm_load_forecast/landing/`.

See `README.md`'s [Forecasting Pipelines](../README.md#forecasting-pipelines) section for the
architecture this notebook exercises stage by stage.

**Stages:** ingest → validate_raw → profile (+ MSTL) → clean → feature_engineer →
validate_features → train (ETS / SARIMAX / GBM) → evaluate_and_register → serve.

**Runtime:** ~10-12 minutes end to end — the profile/MSTL stage alone takes ~8-9 minutes at
full dataset scale (see the "Known limitations" note near the end).

In [1]:
import os
from pathlib import Path

# Notebooks run with CWD=notebooks/ by default; every pipeline path in this repo
# (data/, config/, reports/) is relative to the repo root.
if not Path("pyproject.toml").exists() and Path("../pyproject.toml").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: /Users/danielguo/Development/ai_learning_projects/ml_pipeline


In [2]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import yaml

from src.ingest import ingest_files
from src.validate import validate_raw_files
from src.profile import profile_raw_files, generate_mstl_report
from src.forecasting.clean_forecast import clean_forecast_data
from src.forecasting.features_forecast import engineer_forecast_features
from src.schemas.features import build_forecast_features_schema
from src.forecasting.train_forecast import train_forecast_models
from src.forecasting.evaluate_forecast import register_forecast_models_to_mlflow
from src.forecasting.serve_forecast import forecast_with_gbm, load_latest_snapshot

plt.rcParams["figure.figsize"] = (12, 4)

CONFIG_DIR = "config/pjm_load_forecast"
LANDING_DIR = "data/pjm_load_forecast/landing"
RAW_DIR = "data/pjm_load_forecast/raw"
INTERIM_DIR = "data/pjm_load_forecast/interim"
FEATURES_DIR = "data/pjm_load_forecast/features"
REPORTS_DIR = "reports/pjm_load_forecast"

# A local, self-contained MLflow store for this notebook's own demo run - NOT the
# docker-compose mlflow-server at localhost:5001. mlflow-server's artifact root
# (/mlflow-artifacts) is bind-mounted and writable inside the Airflow/FastAPI
# containers, but a plain host Python process (this notebook's kernel) can't write
# there, so client-side mlflow.*.log_model() calls against the real server fail with
# "Read-only file system: '/mlflow-artifacts'" outside Docker. A local sqlite +
# local-file store keeps this notebook fully reproducible with no dependency on
# docker-compose being up at all - the same pattern this repo's own test suite uses
# (see e.g. tests/test_train_forecast.py's mlflow_uri = f"sqlite:///{tmp_path}/...").
MLFLOW_TRACKING_URI = f"sqlite:///{Path.cwd() / 'notebooks' / 'pjm_load_forecast_demo_mlflow.db'}"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("pjm_load_forecast_notebook_demo")

RUN_ID = "notebook-demo"  # distinct from Airflow's date-based run_ids, so re-running this
                           # notebook never collides with a real scheduled/manual DAG run
print(f"RUN_ID = {RUN_ID!r}")
print(f"MLflow tracking URI = {MLFLOW_TRACKING_URI}")

2026/09/18 11:17:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/18 11:17:33 INFO mlflow.store.db.utils: Updating database tables


2026/09/18 11:17:33 INFO mlflow.tracking.fluent: Experiment with name 'pjm_load_forecast_notebook_demo' does not exist. Creating a new experiment.


RUN_ID = 'notebook-demo'
MLflow tracking URI = sqlite:////Users/danielguo/Development/ai_learning_projects/ml_pipeline/notebooks/pjm_load_forecast_demo_mlflow.db


## Stage 1-2: Ingest & Validate

`ingest_files` copies the landing CSV into a versioned `raw/<run_id>/` directory and writes a
checksummed `manifest.yaml`. `validate_raw_files` then checks it against the pandera-backed rules
in `config/pjm_load_forecast/pipeline.yaml` (`Datetime`/`PJME_MW` required, `PJME_MW` bounded
0-120,000 MW, min 1,000 rows).

In [3]:
ingest_result = ingest_files(LANDING_DIR, RAW_DIR, RUN_ID)
print(f"Ingested {ingest_result['file_count']} file(s) -> {ingest_result['manifest_path']}")

validate_result = validate_raw_files(RAW_DIR, RUN_ID, config_dir=CONFIG_DIR)
print(f"Validated: {list(validate_result['validated_files'])}")
print(f"Failed: {validate_result['failed_files']}")

Ingested 1 file(s) -> data/pjm_load_forecast/raw/notebook-demo/manifest.yaml
Validated: ['PJME_hourly.csv']
Failed: []


## Stage 3: Profile + MSTL seasonal decomposition

`profile_raw_files` generates the standard ydata-profiling HTML report. For forecasting
pipelines, `dag_factory.py`'s `profile_wrapper` additionally calls `generate_mstl_report`
(`src/profile.py`) — a multiple-seasonal-trend decomposition (daily 24h + weekly 168h periods,
annual 8766h only if the series is long enough) that this notebook's raw series comfortably is.

**This is the slowest stage** — statsmodels' STL decomposition scales with both series length and
periods count; ~8-9 minutes at this dataset's ~145k rows. See the README's known-limitation note.

In [4]:
%%time
profile_result = profile_raw_files(RAW_DIR, RUN_ID, reports_dir=REPORTS_DIR, config_dir=CONFIG_DIR)
for fname, info in profile_result["reports"].items():
    print(f"{fname}: {info['rows']:,} rows x {info['columns']} cols -> {info['report_path']}")

mstl_result = generate_mstl_report(RAW_DIR, RUN_ID, reports_dir=REPORTS_DIR, config_dir=CONFIG_DIR)
print(f"MSTL periods used: {mstl_result['periods_used']} -> {mstl_result['report_path']}")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|          | 0/7 [00:00<?, ?it/s, Describe variable: Datetime]

Summarize dataset:   0%|          | 0/7 [00:00<?, ?it/s, Describe variable: PJME_MW] 

  0%|          | 0/2 [00:00<?, ?it/s]

Summarize dataset:  14%|█▍        | 1/7 [00:00<00:01,  5.61it/s, Describe variable: PJME_MW]

...

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 22.87it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 1834.78it/s]

PJME_hourly.csv: 145,366 rows x 2 cols -> reports/pjm_load_forecast/notebook-demo_PJME_hourly_profile.html


MSTL periods used: [24, 168, 8766] -> reports/pjm_load_forecast/notebook-demo_mstl.html
CPU times: user 8min 3s, sys: 6.84 s, total: 8min 10s
Wall time: 8min 4s


A quick illustrative look at the raw series and its seasonal shape (last 90 days, for a readable
plot — the full MSTL report above covers the entire series):

In [5]:
raw_df = pd.read_csv(f"{RAW_DIR}/{RUN_ID}/PJME_hourly.csv", parse_dates=["Datetime"]).sort_values("Datetime")
raw_df = raw_df.set_index("Datetime")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
raw_df["PJME_MW"].plot(ax=axes[0], linewidth=0.5)
axes[0].set_title(f"Full series: {raw_df.index.min().date()} to {raw_df.index.max().date()} ({len(raw_df):,} hours)")
axes[0].set_ylabel("MW")

raw_df["PJME_MW"].tail(24 * 90).plot(ax=axes[1])
axes[1].set_title("Last 90 days — daily/weekly seasonality visible")
axes[1].set_ylabel("MW")
plt.tight_layout()
plt.show()

## Stage 4-6: Clean, Feature Engineering, Validate Features

`clean_forecast_data` sorts chronologically (the raw Kaggle CSV is **not** sorted — a known
quirk of this dataset), drops exact-duplicate timestamps, and fills short gaps (fails loud on
anything longer than `max_gap_hours`, per `config/pjm_load_forecast/cleaning.yaml`).
`engineer_forecast_features` builds lag/rolling/calendar/holiday features, splits chronologically
(never randomly — the most recent slice is always the test set), and snapshots the trailing
`snapshot_hours` rows for serving-time lag seeding.

In [6]:
clean_result = clean_forecast_data(RAW_DIR, INTERIM_DIR, RUN_ID, config_dir=CONFIG_DIR)
print(f"Cleaned: {clean_result['row_count']:,} rows, "
 f"{clean_result['duplicates_dropped']} duplicates dropped, "
 f"{clean_result['gaps_found']} gap(s) filled ({clean_result['gaps_filled_hours']} hours)")

features_result = engineer_forecast_features(INTERIM_DIR, FEATURES_DIR, RUN_ID, config_dir=CONFIG_DIR)
print(f"Train shape: {features_result['train_shape']}, Test shape: {features_result['test_shape']}")
print(f"Feature count: {features_result['feature_count']}, rows dropped to lag warm-up: {features_result['rows_dropped_warmup']}")

train_df = pd.read_parquet(features_result["train_path"])
train_df.head()

Cleaned: 145,392 rows, 4 duplicates dropped, 30 gap(s) filled (30 hours)


Train shape: (116179, 14), Test shape: (29045, 14)
Feature count: 13, rows dropped to lag warm-up: 168


,PJME_MW,lag_1h,lag_24h,lag_168h,rolling_mean_24h,rolling_std_24h,rolling_mean_168h,rolling_std_168h,hour,day_of_week,month,is_weekend,is_holiday,days_to_nearest_holiday
Datetime,,,,,,,,,,,,,,
2002-01-08 01:00:00,29445.0,31187.0,26862.0,30393.0,33452.583333,4559.767709,32519.511905,3857.950565,1,1,1,0,0,7
2002-01-08 02:00:00,28670.0,29445.0,25976.0,29265.0,33560.208333,4425.965952,32513.869048,3861.770954,2,1,1,0,0,7
2002-01-08 03:00:00,28375.0,28670.0,25641.0,28357.0,33672.458333,4256.159403,32510.327381,3865.039821,3,1,1,0,0,7
2002-01-08 04:00:00,28542.0,28375.0,25666.0,27899.0,33786.375000,4064.104959,32510.434524,3864.924245,4,1,1,0,0,7
2002-01-08 05:00:00,29261.0,28542.0,26328.0,28057.0,33906.208333,3851.076461,32514.261905,3860.646270,5,1,1,0,0,7


In [7]:
# Mirrors dag_factory.py's validate_features_wrapper (a closure, not a standalone function):
# row-count floor, all-numeric guard, and the forecasting DatetimeIndex pandera schema.
assert len(train_df) >= 100, "train set too small"
non_numeric = train_df.select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"non-numeric columns: {non_numeric}"
build_forecast_features_schema("PJME_MW").validate(train_df)
print(f"validate_features passed: {len(train_df):,} rows, all-numeric, schema-valid")

validate_features passed: 116,179 rows, all-numeric, schema-valid


## Stage 7: Train — ETS, SARIMAX, GBM via rolling-origin cross-validation

Three model families, all compared on the same rolling-origin CV footing
(`src/forecasting/rolling_origin.py`): ETS and SARIMAX are fit once and scored via statsmodels'
dynamic `get_prediction`; GBM is scored via the shared `recursive_forecast` helper. All log
`cv_mape_mean`/`cv_mape_std` to MLflow.

SARIMAX trains on a `max_train_hours`-capped trailing window (`config/pjm_load_forecast/models.yaml`)
with `simple_differencing=True` — both added after this pipeline's first full-scale run got
OOM-killed; see "Known limitations" below.

In [8]:
%%time
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
train_result = train_forecast_models(FEATURES_DIR, RUN_ID, config_dir=CONFIG_DIR, mlflow_tracking_uri=MLFLOW_TRACKING_URI)

cv_summary = pd.DataFrame({
 name: {"cv_mape_mean": info["cv_mape_mean"], "cv_mape_std": info["cv_mape_std"], "n_windows_scored": info["n_windows_scored"]}
 for name, info in train_result["models"].items()
}).T
cv_summary

/Users/danielguo/Development/ai_learning_projects/ml_pipeline/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency H will be used.
  self._init_dates(dates, freq)


2026/09/18 11:25:41 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/danielguo/Development/ai_learning_projects/ml_pipeline


2026/09/18 11:25:41 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/danielguo/Development/ai_learning_projects/ml_pipeline


2026/09/18 11:25:41 INFO mlflow.utils.environment: Detected uv project at /Users/danielguo/Development/ai_learning_projects/ml_pipeline. Attempting to export requirements via 'uv export'.


2026/09/18 11:25:41 INFO mlflow.utils.uv_utils: Exported 251 dependencies via uv


...

2026/09/18 11:26:29 INFO mlflow.utils.uv_utils: Exported 251 dependencies via uv


2026/09/18 11:26:29 INFO mlflow.utils.environment: Successfully exported 251 requirements from uv project. Skipping package capture based inference.


2026/09/18 11:26:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CPU times: user 1min 29s, sys: 11min, total: 12min 29s
Wall time: 51.6 s


,cv_mape_mean,cv_mape_std,n_windows_scored
pjm_load_forecast_ets,11.995663,3.936542,5.0
pjm_load_forecast_sarimax,100.031573,0.047428,5.0
pjm_load_forecast_gbm,3.193780,0.791956,5.0


In [9]:
cv_summary["cv_mape_mean"].sort_values().plot.barh(xerr=cv_summary["cv_mape_std"], color="steelblue")
plt.xlabel("CV MAPE (%) — lower is better")
plt.title("Training-set rolling-origin CV by model")
plt.tight_layout()
plt.show()

## Stage 8: Evaluate & Register to MLflow

Each trained model is reloaded from MLflow by its logged flavor and scored again — this time at
origins drawn from the held-out **test** period, the genuine out-of-sample generalization
estimate. Passing models register to MLflow `Staging` (never auto-promoted to `Production` — that
stays a manual step, same as every tabular pipeline in this repo). The run champion is whichever
model has the lowest `cv_mape`.

In [10]:
run_ids = {name: info["mlflow_run_id"] for name, info in train_result["models"].items()}
register_result = register_forecast_models_to_mlflow(
 mlflow_tracking_uri=MLFLOW_TRACKING_URI, mlflow_run_ids=run_ids, config_dir=CONFIG_DIR,
 run_id=RUN_ID, reports_dir=REPORTS_DIR, features_dir=FEATURES_DIR,
)

test_summary = pd.DataFrame({
 name: {"status": info["status"], "test_mape_mean": info.get("test_mape_mean"), "cv_mape_mean": info.get("cv_mape_mean")}
 for name, info in register_result["registered_models"].items()
}).T
test_summary

Successfully registered model 'pjm_load_forecast_ets'.
2026/09/18 11:26:30 WARNING mlflow.tracking._model_registry.fluent: Run with id cbfca2cfa7814f4080f517fa9563d105 has no artifacts at artifact path 'model', registering model based on models:/m-9506966bacdf4ed290e8dc6fe67305cf instead


Created version '1' of model 'pjm_load_forecast_ets'.


Successfully registered model 'pjm_load_forecast_sarimax'.
2026/09/18 11:26:32 WARNING mlflow.tracking._model_registry.fluent: Run with id dcd052c6922e41d09fe1147b5e4cd153 has no artifacts at artifact path 'model', registering model based on models:/m-6ecf14acdaa741dfbc677a0312070cfa instead


Created version '1' of model 'pjm_load_forecast_sarimax'.


Successfully registered model 'pjm_load_forecast_gbm'.
2026/09/18 11:26:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 343ce792ee824e269bae8384621df07b has no artifacts at artifact path 'model', registering model based on models:/m-a12c7055cd8142d9a8a2fa83c61668cd instead


Created version '1' of model 'pjm_load_forecast_gbm'.


,status,test_mape_mean,cv_mape_mean
pjm_load_forecast_ets,registered,21.135529,11.995663
pjm_load_forecast_sarimax,registered,99.940663,100.031573
pjm_load_forecast_gbm,registered,3.018263,3.19378


In [11]:
with open(f"{REPORTS_DIR}/{RUN_ID}_evaluation.yaml") as f:
 evaluation_report = yaml.safe_load(f)
champion = evaluation_report["run_champion"]
print(f"Run champion: {champion} (by {evaluation_report['champion_metric']})")

Run champion: pjm_load_forecast_gbm (by cv_mape)


## Stage 9: Serving — a live multi-step forecast

Loads the champion model straight back from MLflow (the same `mlflow.lightgbm.load_model` /
`mlflow.statsmodels.load_model` flavor dispatch `src/serve.py`'s `_load_model` uses) and produces
a real forecast via `forecast_with_gbm`, seeded from the run's own `last_window.parquet` snapshot
— the same function backing the live `GET /predict/forecast` endpoint.

In [12]:
import mlflow.lightgbm

champion_run_id = run_ids[champion]
champion_model = mlflow.lightgbm.load_model(f"runs:/{champion_run_id}/model")

with open(f"{CONFIG_DIR}/features.yaml") as f:
 features_cfg = yaml.safe_load(f)
with open(f"{CONFIG_DIR}/pipeline.yaml") as f:
 pipeline_cfg = yaml.safe_load(f)
target_col = pipeline_cfg["target"]["name"]
feature_columns = [c for c in train_df.columns if c != target_col]

snapshot = load_latest_snapshot(FEATURES_DIR)
forecast = forecast_with_gbm(
 champion_model, snapshot, target_col, feature_columns, horizon_hours=24,
 lags=features_cfg["lags"], rolling_windows=features_cfg["rolling_windows"],
 calendar_features=features_cfg["calendar_features"], holiday_features=features_cfg["holiday_features"],
)
forecast

Datetime
2018-08-03 01:00:00    32531.208289
2018-08-03 02:00:00    30534.048432
2018-08-03 03:00:00    29116.449242
2018-08-03 04:00:00    28431.712468
2018-08-03 05:00:00    28360.884019
2018-08-03 06:00:00    29737.725812
2018-08-03 07:00:00    32131.604532
2018-08-03 08:00:00    34906.227404
2018-08-03 09:00:00    37220.401833
2018-08-03 10:00:00    39368.477832
2018-08-03 11:00:00    41495.025001
2018-08-03 12:00:00    43062.239349
2018-08-03 13:00:00    44283.058235
2018-08-03 14:00:00    45177.861140
2018-08-03 15:00:00    46148.594147
2018-08-03 16:00:00    46349.739245
2018-08-03 17:00:00    46479.289296
2018-08-03 18:00:00    46034.135708
2018-08-03 19:00:00    44433.809837
2018-08-03 20:00:00    43111.847690
2018-08-03 21:00:00    42313.984567
2018-08-03 22:00:00    40997.504237
2018-08-03 23:00:00    37960.010033
2018-08-04 00:00:00    34922.146584
dtype: float64

In [13]:
fig, ax = plt.subplots()
snapshot[target_col].tail(24 * 7).plot(ax=ax, label="last 7 days (actual)")
forecast.plot(ax=ax, label="24h forecast", linestyle="--", marker="o", markersize=3)
ax.axvline(snapshot.index[-1], color="gray", linestyle=":", linewidth=1)
ax.set_ylabel("MW")
ax.set_title(f"{champion} — 24-hour forecast from {snapshot.index[-1]}")
ax.legend()
plt.tight_layout()
plt.show()

**Note on this notebook's MLflow store:** the training/registration above uses a local sqlite +
local-file MLflow store scoped to this notebook (`notebooks/pjm_load_forecast_demo_mlflow.db`),
not the docker-compose `mlflow-server` at http://localhost:5001 — see the setup cell's comment
for why. This keeps the notebook fully reproducible without Docker running, but it also means
this run's models are separate from whatever is registered in the live deployment.

To serve a model trained this way live over HTTP, promote it to Production in the **real**
mlflow-server's registry the normal way (train via the Airflow DAG or `train_forecast_models`
pointed at `http://mlflow-server:5000`/`http://localhost:5001`, not this notebook's local store),
then:
```bash
# In the MLflow UI (http://localhost:5001): promote the desired version to Production
# In .env: SERVING_MODEL_NAME=pjm_load_forecast_gbm
docker compose up -d fastapi
curl "http://localhost:8000/predict/forecast?horizon_hours=24"
```

## Known limitations (found running this pipeline end-to-end for the first time)

1. **MSTL runtime at full scale** — the profile stage's MSTL decomposition takes ~8-9 minutes
 against the full ~145k-row dataset. No `execution_timeout` is set for this task; acceptable
 for this POC's `@weekly` schedule, documented in the README's known-limitations note.
2. **SARIMAX memory at full scale** — fitting `order=(2,1,2)`/`seasonal_order=(1,1,1,24)` on the
 full ~116k-row training set peaked at **11GB** RSS (measured directly) and OOM-killed this
 project's local Docker deployment (7.75GB VM, shared across 8 services). Fixed two ways:
 `max_train_hours` caps SARIMAX to its trailing 2 years (`config/pjm_load_forecast/models.yaml`)
 while ETS/GBM still train on the full series, and `simple_differencing=True`
 (`src/forecasting/model_registry.py`) dropped that 2-year fit from 5.7GB/non-converged to
 2.2GB/converged.
3. **SARIMAX forecast quality** — even after the fixes above, SARIMAX's registered `test_mape`
 is ~100% (effectively unusable), while GBM and ETS are well-behaved. Root cause not yet
 isolated — a real open item, not a pipeline bug (the pipeline completed correctly; champion
 selection correctly ignored SARIMAX in favor of GBM).

## Summary

| Model | CV MAPE | Test MAPE | Registered |
|---|---|---|---|
| **GBM** (champion) | ~3.2% | ~3.0% | ✅ Staging |
| ETS | ~12.0% | ~21.1% | ✅ Staging |
| SARIMAX | ~100% | ~100% | ✅ Staging (unusable — see limitations above) |

GBM is the clear production candidate: promote it to Production in the MLflow UI
(http://localhost:5001), point `SERVING_MODEL_NAME` at it in `.env`, and
`docker compose up -d fastapi` to serve it live.